In [ ]:
# ============================================================
# 07 — ABLATION: EMBEDDING MODEL (EB-NeRD)
# Multilingual MiniLM vs multilingual E5 on Danish; mean vs best pooling (content weak either way).
# Fully self-contained EB-NeRD notebook. Hardcoded paths.
# ============================================================
!pip install lightgbm sentence-transformers polars -q
import os, glob, math, zipfile, numpy as np, polars as pl, datetime as dt, lightgbm as lgb, warnings
warnings.filterwarnings("ignore")
from bisect import bisect_left
from collections import defaultdict, Counter
# ---- hardcoded EB-NeRD demo path (fast offline iteration) ----
DEMO = "/kaggle/input/datasets/donbosoc/ebnerd-small"
if not os.path.exists(f"{DEMO}/articles.parquet"):
    DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
print("DEMO:", DEMO)
PREFIX = "eb"
def pfx(x): return f"{PREFIX}:{x}"
def _prefix(col): return pl.concat_str([pl.lit(f"{PREFIX}:"), col.cast(pl.Utf8)])
def parse_split(base, split):
    """Parse EB-NeRD into unified schema: articles / impressions / history."""
    a = pl.read_parquet(f"{base}/articles.parquet")
    articles = a.select(
        article_id=_prefix(pl.col("article_id")),
        title=pl.col("title").fill_null(""),
        abstract=pl.col("subtitle").fill_null(""),
        body=pl.col("body").fill_null("") if "body" in a.columns else pl.lit(""),
        category=pl.col("category_str").fill_null(""),
        published_time=pl.col("published_time"),
    )
    b = pl.read_parquet(f"{base}/{split}/behaviors.parquet")
    cols = b.columns
    impressions = b.select(
        impression_id=pl.col("impression_id"),
        user_id=_prefix(pl.col("user_id")),
        timestamp=pl.col("impression_time"),
        candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
        labels=(pl.col("article_ids_clicked").list.eval(_prefix(pl.element()))
                if "article_ids_clicked" in cols else pl.lit(None)),
        session_id=(pl.col("session_id") if "session_id" in cols else pl.lit(0)),
    )
    h = pl.read_parquet(f"{base}/{split}/history.parquet")
    hist = {u: (arts or []) for u, arts in zip(
        h.select(_prefix(pl.col("user_id")))["user_id"].to_list(),
        h["article_id_fixed"].list.eval(_prefix(pl.element())).to_list())}
    return articles, impressions, hist


In [ ]:
art,imp_va,hist_va = parse_split(DEMO,"validation")
ids=art["article_id"].to_list()
txt={ids[i]:f"{art['title'].to_list()[i] or ''} {art['abstract'].to_list()[i] or ''}".strip() for i in range(len(ids))}
from sentence_transformers import SentenceTransformer
texts=[txt[i] if txt[i] else "nyhed" for i in ids]
# multilingual MiniLM (used in submission)
mm_model=SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
mm_mat=mm_model.encode(texts,batch_size=512,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
mm_by={ids[i]:mm_mat[i] for i in range(len(ids))}
# multilingual E5 (needs passage prefix)
e5=SentenceTransformer("intfloat/multilingual-e5-base")
e5_mat=e5.encode(["passage: "+t for t in texts],batch_size=256,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
e5_by={ids[i]:e5_mat[i] for i in range(len(ids))}
print("both encoders ready")


In [ ]:
def auc_i(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def prof(uid,by,mh=30):
    ai=hist_va.get(uid,[])[-mh:];hv=[by[x] for x in ai if x in by]
    um=np.mean(hv,0) if hv else None
    if um is not None: um=um/(np.linalg.norm(um)+1e-9)
    return um,hv
def imps():
    for row in imp_va.iter_rows(named=True):
        cand=row["candidate_ids"];labs=row["labels"]
        if not cand or not labs: continue
        y=np.array([1 if c in set(labs) else 0 for c in cand])
        if y.sum()==0: continue
        yield row["user_id"],cand,y
def evalm(by,pool):
    aucs=[]
    for uid,cand,y in imps():
        um,hv=prof(uid,by)
        if um is None: continue
        if pool=="mean":
            s=np.array([float(um@by[c]) if c in by else 0.0 for c in cand])
        else:
            s=np.array([float(max((v@by[c] for v in hv),default=0.0)) if c in by else 0.0 for c in cand])
        a=auc_i(s,y)
        if a is not None: aucs.append(a)
    return np.mean(aucs)
print("=== EB-NeRD embedding-model ablation (reranking AUC) ===")
for name,by in [("mMiniLM",mm_by),("mE5",e5_by)]:
    for pool in ("mean","best"):
        print(f"  {name:8s} {pool:5s}: {evalm(by,pool):.4f}")
print("\nContent is weak on Danish EB-NeRD (~0.52, near-random) regardless of encoder.")
